<a href="https://colab.research.google.com/github/MANIKANTH678/DopamineOS/blob/main/entity_resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Business entity resolution: S1 ↔ S2 / S3

Matches each `source1` business to its records in `source2` and `source3`.
This replaces the exploratory `Untitled1.ipynb` and is meant to be run top to bottom (**Runtime → Run all**).

| Stage | What it does | Cost |
|---|---|---|
| 2. Normalize | DuckDB reads the TSVs, transliterates non-Latin names (Devanagari, Tamil, Telugu, Bengali, accents), strips legal suffixes, writes Parquet to Drive | once; cached |
| 3. Blocking | Keys built from each record's *rarest* name/address tokens; keys whose block would produce more than `MAX_KEY_PAIRS` pairs are dropped | seconds–minutes |
| 4. Training | Candidates only for a sample of S1 entities; features computed in C with `rapidfuzz.process.cpdist`; `HistGradientBoostingClassifier` | minutes |
| 5. Threshold | Pick the F-beta-optimal threshold on held-out S1 entities; recall lost to blocking is counted as missed | seconds |
| 6. Test | Same blocking + features in streamed batches, write predictions | minutes |

Why the old notebook never finished: its blocking keys (`country|state`, `country|last token`, …) put most rows of a
country into one block, so the S1×S2 join was close to a full cross join (an estimated ~10¹² pairs) run as a SQLite nested loop.
Here every block's pair count is computed **before** the join and capped, and the join runs as a parallel hash join.

**If a cell fails:** fix the cause and use *Run all* again rather than adding repair cells. The notebook stops by
itself if its normalization code or the Parquet cache is wrong, and it keeps its DuckDB database on local disk
(never put a `.duckdb` file on Google Drive; it gets corrupted there).

## 1. Setup

In [ ]:
import sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'duckdb>=1.1', 'rapidfuzz>=3.6', 'anyascii', 'pyarrow', 'scikit-learn'], check=True)

In [ ]:
import os, shutil, time
from contextlib import contextmanager
from pathlib import Path

import duckdb
import joblib
import numpy as np
import pandas as pd
import pyarrow as pa
from anyascii import anyascii
from rapidfuzz import fuzz, process
from rapidfuzz.distance import JaroWinkler
from sklearn.ensemble import HistGradientBoostingClassifier

# ---- paths (env vars let the same notebook run outside Colab) ----
DATA_ZIP  = Path(os.environ.get('ER_DATA_ZIP', '/content/drive/MyDrive/dataset.zip'))
DATA_DIR  = Path(os.environ.get('ER_DATA_DIR', '/content/dataset'))             # where the zip is extracted
CACHE_DIR = Path(os.environ.get('ER_CACHE_DIR',                                 # survives runtime resets
                 '/content/drive/MyDrive/er_cache' if IN_COLAB else 'er_cache'))
WORK_DIR  = Path(os.environ.get('ER_WORK_DIR', '/content/er_work' if IN_COLAB else 'er_work'))  # fast local disk

# ---- knobs ----
MAX_KEY_PAIRS    = 1_000    # drop a blocking key if (#S1 rows) × (#S2+S3 rows) sharing it exceeds this
MAX_CANDS_PER_S1 = 200      # keep at most this many candidates per S1 record (most shared keys first)
N_TRAIN_S1       = 200_000  # S1 entities used for training (all 2.2M are not needed)
N_VAL_S1         = 50_000   # held-out S1 entities for recall / threshold selection
MAX_TRAIN_ROWS   = 4_000_000  # negatives are subsampled above this
BETA             = 1.0      # set to the beta of the competition's F-beta metric
SEED             = 42
BATCH_ROWS       = 250_000  # candidate pairs featurized per batch
REBUILD          = False    # True: ignore cached normalized Parquet
DUCKDB_MEMORY    = os.environ.get('ER_DUCKDB_MEMORY', '6GB')  # DuckDB limit for blocking (Colab VM: 12.7 GB)
DUCKDB_STREAM_MEMORY = os.environ.get('ER_DUCKDB_STREAM_MEMORY', '2GB')  # while featurizing: RAM goes to Python
DUCKDB_THREADS   = int(os.environ.get('ER_DUCKDB_THREADS', os.cpu_count()))

# A DuckDB database file on Google Drive gets corrupted ("Could not read enough bytes ..."): Drive is not a
# real disk. Only the Parquet cache and outputs go to Drive; the database stays on local disk.
assert not str(WORK_DIR.resolve()).startswith('/content/drive'), 'WORK_DIR must be on local disk, not Google Drive'

WORK_DIR.mkdir(parents=True, exist_ok=True)  # CACHE_DIR is created after Drive is mounted


@contextmanager
def stage(name):
    t = time.time()
    print(f'▶ {name}')
    yield
    print(f'✔ {name}: {time.time() - t:,.1f}s')


def save_to_cache(local_path):
    """Copy a finished file from local disk to CACHE_DIR (Drive). A full Drive must not crash the run."""
    dest = CACHE_DIR / local_path.name
    try:
        shutil.copyfile(local_path, dest)
        return dest
    except OSError as e:
        dest.unlink(missing_ok=True)  # never leave a truncated copy behind
        print(f'WARNING: could not save {local_path.name} to {CACHE_DIR}: {e}\n'
              f'  It is still available at {local_path} (lost when the runtime resets).\n'
              '  If Google Drive is full: delete er_checkpoint/ and other large files, then EMPTY THE DRIVE TRASH '
              '(deleted files keep using the quota until the trash is emptied).')
        return local_path

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

CACHE_DIR.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(CACHE_DIR).free / 2**30
print(f'free space for {CACHE_DIR}: {free_gb:,.1f} GB')
if free_gb < 5:
    print('WARNING: less than 5 GB free where the cache is written (it needs about 4 GB). On Google Drive, delete '
          'er_checkpoint/ and other large files, then empty the Drive trash.')

if not any(p for p in DATA_DIR.rglob('train_ground_truth.tsv') if '__MACOSX' not in p.parts):
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with stage('unzip dataset'):
        subprocess.run(['unzip', '-q', '-o', str(DATA_ZIP), '-d', str(DATA_DIR)], check=True)

GT_PATH = next(p for p in DATA_DIR.rglob('train_ground_truth.tsv') if '__MACOSX' not in p.parts)
BASE = GT_PATH.parent.parent
SOURCES = {split: {s: BASE / split / f'{split}_source{s}.tsv' for s in (1, 2, 3)} for split in ('train', 'test')}
for split, files in SOURCES.items():
    for s, p in files.items():
        assert p.exists(), p
        print(f'{p.name:20s} {p.stat().st_size / 2**30:5.2f} GB')

readme = BASE.parent / 'README.md'
if readme.exists():
    print(readme.read_text()[:3000])  # check the submission format described here

In [ ]:
# The whole notebook uses this one connection. Do not replace it with a new duckdb.connect() in a later
# cell: that drops the to_ascii function, the spill directory and the memory limit set here.
con = duckdb.connect(str(WORK_DIR / 'er.duckdb'))
con.execute(f"SET temp_directory = '{WORK_DIR / 'spill'}'")
con.execute(f"SET memory_limit = '{DUCKDB_MEMORY}'")
con.execute(f'SET threads = {DUCKDB_THREADS}')
con.execute('SET preserve_insertion_order = false')
print('duckdb', duckdb.__version__, '| threads:', con.execute("SELECT current_setting('threads')").fetchone()[0])


def _to_ascii(col):
    # anyascii handles Devanagari/Tamil/Telugu/Bengali and accents: 'मॉडर्न फाइनेंस' -> 'modrn phainems'
    return pa.array([anyascii(s) if s is not None and not s.isascii() else s for s in col.to_pylist()],
                    type=pa.string())


con.create_function('to_ascii', _to_ascii, ['VARCHAR'], 'VARCHAR', type='arrow')

## 2. Normalize sources (cached as Parquet)

* `name_norm` – lower-case ASCII alphanumerics.
* `name_core` – `name_norm` without legal forms, honorifics and articles (these made the old
  first/last-token blocks enormous).
* `name_skel` – rough phonetic skeleton of `name_core`, so transliterated names (`marketimg`) still resemble
  English ones (`marketing`).
* `addr_norm` – address with street-type abbreviations expanded; `addr_nums` – sorted distinct numbers in it.

In [ ]:
LEGAL_AND_FILLER = [
    # legal forms (incl. common transliterations)
    'private', 'pvt', 'praivet', 'limited', 'ltd', 'limitad', 'llp', 'llc', 'l l c', 'inc', 'incorporated',
    'corp', 'corporation', 'co', 'company', 'plc', 'opc', 'sarl', 'sas', 'sasu', 'sa', 'eurl', 'sci', 'snc',
    'gmbh', 'pllc', 'lp', 'ltda',
    # honorifics / prefixes
    'm s', 'shri', 'shree', 'sri', 'sree', 'shrii', 'smt',
    # articles / connectors
    'the', 'and', 'of', 'et', 'le', 'la', 'les', 'de', 'du', 'des',
]
ADDR_ABBR = {
    'rd': 'road', 'st': 'street', 'ave': 'avenue', 'av': 'avenue', 'blvd': 'boulevard', 'bd': 'boulevard',
    'dr': 'drive', 'ln': 'lane', 'ct': 'court', 'hwy': 'highway', 'pkwy': 'parkway', 'pl': 'place',
    'sq': 'square', 'ter': 'terrace', 'apt': 'apartment', 'ste': 'suite', 'fl': 'floor', 'bldg': 'building',
    'nagr': 'nagar', 'mkt': 'market', 'opp': 'opposite',
}


def sql_clean(col):
    """SQL expression: lower-case ASCII tokens separated by single spaces."""
    ascii_ = (f"CASE WHEN regexp_matches(coalesce({col}, ''), '[^\\x00-\\x7F]') "
              f"THEN to_ascii({col}) ELSE coalesce({col}, '') END")
    return f"trim(regexp_replace(replace(lower({ascii_}), '&', ' and '), '[^a-z0-9]+', ' ', 'g'))"


def sql_squash(expr):
    return f"trim(regexp_replace({expr}, ' +', ' ', 'g'))"


STOP_RE = r'\b(' + '|'.join(LEGAL_AND_FILLER) + r')\b'


def sql_core(expr):
    return sql_squash(f"regexp_replace({expr}, '{STOP_RE}', ' ', 'g')")


def sql_skeleton(expr):
    e = f"regexp_replace({expr}, 'ph', 'f', 'g')"
    e = f"regexp_replace({e}, 'w', 'v', 'g')"
    e = f"regexp_replace({e}, 'm([kgcjtdsz])', 'n\\1', 'g')"   # anusvara transliterated as 'm'
    return f"regexp_replace({e}, '\\B[aeiouy]', '', 'g')"         # drop non-initial vowels


def sql_addr(expr):
    for short, full in ADDR_ABBR.items():
        expr = f"regexp_replace({expr}, '\\b{short}\\b', '{full}', 'g')"
    return expr


def build_recs(split):
    """Create table recs_<split> (one row per entity, all three sources), cached as Parquet."""
    pq = CACHE_DIR / f'recs_{split}.parquet'
    if pq.exists() and not REBUILD:
        con.execute(f"CREATE OR REPLACE TABLE recs_{split} AS SELECT * FROM read_parquet('{pq}')")
        return
    raw = ' UNION ALL '.join(
        f"SELECT {s}::TINYINT AS src, * FROM read_csv('{path}', delim='\\t', header=true, all_varchar=true, "
        f"quote='\"', escape='\"')"
        for s, path in SOURCES[split].items())
    con.execute(f"""
        CREATE OR REPLACE TABLE recs_{split} AS
        WITH raw AS ({raw}),
        n AS (
            SELECT src, trim(entity_id) AS entity_id,
                   lower(trim(coalesce(country, ''))) AS country,
                   business_name AS name_raw, business_address AS addr_raw,
                   regexp_matches(coalesce(business_name, ''), '[^\\x00-\\x7F]') AS non_ascii,
                   {sql_clean('business_name')} AS name_norm,
                   {sql_squash(sql_addr(sql_clean('business_address')))} AS addr_norm
            FROM raw
        ),
        c AS (SELECT *, {sql_core('name_norm')} AS name_core FROM n)
        SELECT *, {sql_skeleton('name_core')} AS name_skel,
               array_to_string(list_sort(list_distinct(regexp_extract_all(addr_norm, '[0-9]+'))), ' ') AS addr_nums
        FROM c
        QUALIFY row_number() OVER (PARTITION BY entity_id ORDER BY src) = 1
    """)
    # guard against silent row loss (stray quotes) or duplicate ids, which corrupted the old SQLite tables
    got = dict(con.execute(f'SELECT src, count(*) FROM recs_{split} GROUP BY src').fetchall())
    for s, path in SOURCES[split].items():
        with open(path, 'rb') as f:
            lines = sum(1 for _ in f) - 1
        if got.get(s, 0) != lines:
            print(f'WARNING {path.name}: {lines:,} data lines but {got.get(s, 0):,} unique records loaded')
    local_pq = WORK_DIR / pq.name
    con.execute(f"COPY recs_{split} TO '{local_pq}' (FORMAT parquet)")
    if save_to_cache(local_pq) != local_pq:
        local_pq.unlink()


# ---- self-checks: fail in seconds instead of producing silently wrong data for hours ----
NAME_CASES = {  # raw name -> (name_core, name_skel, non_ascii)
    'M/S Producer SNK Infrastructure Pvt. Ltd.': ('producer snk infrastructure', 'prdcr snk infrstrctr', False),
    "Stephenie's Seafood": ('stephenie s seafood', 'stfn s sfd', False),
    'मॉडर्न फाइनेंस': ('modrn phainems', 'mdrn fnns', True),
    'Café de la Gare SARL': ('cafe gare', 'cf gr', True),
    'Shree Ram Marketing Private Limited': ('ram marketing', 'rm mrktng', False),
}
ADDR_CASES = {  # raw address -> addr_norm
    '00719 Lincoln Ave, Carrollton, OH': '00719 lincoln avenue carrollton oh',
    'Plot No. 1, Opp. Bus Stand, Nehru Rd, Vadodara': 'plot no 1 opposite bus stand nehru road vadodara',
}
FIX_HINT = ('The normalization helpers in this notebook were modified (a common cause is doubled regex '
            'backslashes). Re-download entity_resolution.ipynb instead of patching the SQL.')


def sql_values(strings):
    return 'VALUES ' + ', '.join("('" + s.replace("'", "''") + "')" for s in strings)


def check_normalizers():
    got = con.execute(f"""
        SELECT x, {sql_core(sql_clean('x'))}, {sql_skeleton(sql_core(sql_clean('x')))},
               regexp_matches(x, '[^\\x00-\\x7F]')
        FROM ({sql_values(NAME_CASES)}) t(x)
    """).fetchall()
    got += [(x, a) for x, a in con.execute(
        f"SELECT x, {sql_squash(sql_addr(sql_clean('x')))} FROM ({sql_values(ADDR_CASES)}) t(x)").fetchall()]
    expected = {**{k: (k, *v) for k, v in NAME_CASES.items()}, **{k: (k, v) for k, v in ADDR_CASES.items()}}
    bad = [f'  {row[0]!r}: got {row[1:]}, expected {expected[row[0]][1:]}' for row in got if row != expected[row[0]]]
    assert not bad, FIX_HINT + '\n' + '\n'.join(bad)
    print('✔ normalization self-check passed')


def check_recs(split, sample=20_000):
    """Recompute a sample of recs_<split> with the current helpers; catches a stale or broken Parquet cache."""
    bad = con.execute(f"""
        SELECT count(*) FILTER (WHERE name_norm <> {sql_clean('name_raw')}) AS name_norm,
               count(*) FILTER (WHERE name_core <> {sql_core('name_norm')}) AS name_core,
               count(*) FILTER (WHERE name_skel <> {sql_skeleton('name_core')}) AS name_skel,
               count(*) FILTER (WHERE addr_norm <> {sql_squash(sql_addr(sql_clean('addr_raw')))}) AS addr_norm,
               count(*) FILTER (WHERE non_ascii <> regexp_matches(coalesce(name_raw, ''), '[^\\x00-\\x7F]')) AS non_ascii
        FROM (SELECT * FROM recs_{split} USING SAMPLE {sample} ROWS)
    """).df()
    assert bad.to_numpy().sum() == 0, (
        f'recs_{split} does not match this notebook\'s normalization (mismatching rows per column below). '
        f'Delete {CACHE_DIR / f"recs_{split}.parquet"} or set REBUILD = True, then rerun.\n{bad.to_string(index=False)}')
    print(f'✔ recs_{split} matches the normalization code')


check_normalizers()

In [ ]:
with stage('normalize train'):
    build_recs('train')
with stage('normalize test'):
    build_recs('test')

for split in ('train', 'test'):
    check_recs(split)
    print(split, con.execute(f'SELECT src, count(*) FROM recs_{split} GROUP BY src ORDER BY src').fetchall())
con.sql("""SELECT src, name_raw, name_norm, name_core, name_skel, addr_norm, addr_nums
           FROM recs_train USING SAMPLE 8 ROWS""").df()

## 3. Blocking

For every record we pick the two **rarest** tokens of `name_core` and of the address (by document frequency
within the country), and build keys from them:

| key | built from | example |
|---|---|---|
| `n1` | each of the two rarest name tokens alone | `india\|roongta` |
| `n2` | the two rarest name tokens together | `india\|roongta sangh` |
| `px` | first 6 chars of the core name | `india\|roongt` |
| `sk` | first 8 chars of the phonetic skeleton (survives vowel typos: `eldorsa`/`elodrsa` → `eldrs`) | `india\|rngtsngh` |
| `a2` | the two rarest address tokens together | `india\|bhubaneswar mahodadhi` |
| `na` | each rarest name token × each rarest address token (4 keys; survives a typo in one token) | `india\|roongta\|mahodadhi` |

Keys shared by too many records (`n_s1 × n_s23 > MAX_KEY_PAIRS`) are dropped, which bounds the number of
candidates before any join runs.

In [ ]:
def rare_tokens(split, text_col, out, min_len):
    """out(entity_id, r1, r2): the two rarest tokens of text_col (ties broken alphabetically).

    At full size this handles ~125M (record, token) pairs on a 2-core Colab VM. Every heavy step works on
    integer ids with fixed-size aggregates so DuckDB can spill to disk; a window over the token rows was slow,
    and per-record list/string aggregates ran out of memory.
    """
    tokens = f"""
        SELECT e.eid, e.country, unnest(string_split(r.{text_col}, ' ')) AS tok
        FROM recs_{split} r JOIN ents_{split} e USING (entity_id)"""
    token_filter = f"length(tok) >= {min_len} AND NOT regexp_matches(tok, '^[0-9]+$')"
    shift = 2 ** 31  # key = df * shift + tid orders by document frequency, then alphabetically (tid follows tok)
    con.execute(f"""
        CREATE OR REPLACE TABLE _rt_dict AS
        SELECT country, tok, (row_number() OVER (ORDER BY tok, country))::INTEGER AS tid
        FROM (SELECT DISTINCT country, tok FROM ({tokens}) WHERE {token_filter})
    """)
    con.execute(f"""
        CREATE OR REPLACE TABLE _rt_et AS
        SELECT DISTINCT t.eid, d.tid FROM ({tokens}) t JOIN _rt_dict d USING (country, tok)
    """)
    con.execute(f"""
        CREATE OR REPLACE TABLE _rt_key AS
        WITH df AS (SELECT tid, count(*) AS df FROM _rt_et GROUP BY tid)
        SELECT et.eid, df.df * {shift} + et.tid AS key FROM _rt_et et JOIN df USING (tid)
    """)
    con.execute('CREATE OR REPLACE TABLE _rt_m1 AS SELECT eid, min(key) AS k1 FROM _rt_key GROUP BY eid')
    con.execute("""
        CREATE OR REPLACE TABLE _rt_m2 AS
        SELECT k.eid, min(k.key) AS k2 FROM _rt_key k JOIN _rt_m1 m USING (eid) WHERE k.key > m.k1 GROUP BY k.eid
    """)
    con.execute(f"""
        CREATE OR REPLACE TABLE {out} AS
        SELECT e.entity_id, d1.tok AS r1, d2.tok AS r2
        FROM _rt_m1 m1
        JOIN ents_{split} e USING (eid)
        JOIN _rt_dict d1 ON d1.tid = m1.k1 % {shift}
        LEFT JOIN _rt_m2 m2 USING (eid)
        LEFT JOIN _rt_dict d2 ON d2.tid = m2.k2 % {shift}
    """)
    for t in ('_rt_dict', '_rt_et', '_rt_key', '_rt_m1', '_rt_m2'):
        con.execute(f'DROP TABLE {t}')


KEY_TYPES = ['n1', 'n2', 'px', 'sk', 'a2', 'na']


def build_keys(split):
    con.execute(f'''CREATE OR REPLACE TABLE ents_{split} AS
                    SELECT (row_number() OVER ())::INTEGER AS eid, entity_id, country FROM recs_{split}''')
    with stage(f'rarest name tokens ({split})'):
        rare_tokens(split, 'name_core', f'nrare_{split}', 2)
    with stage(f'rarest address tokens ({split})'):
        rare_tokens(split, 'addr_norm', f'arare_{split}', 3)
    t = time.time()
    # every key carries its type prefix and r1 != r2, a1 != a2, so rows are already distinct (no DISTINCT)
    con.execute(f"""
        CREATE OR REPLACE TABLE keys_{split} AS
        WITH b AS (
            SELECT r.entity_id, r.src, r.country, replace(r.name_core, ' ', '') AS core_ns,
                   replace(r.name_skel, ' ', '') AS skel_ns, n.r1 AS n1, n.r2 AS n2, a.r1 AS a1, a.r2 AS a2
            FROM recs_{split} r
            LEFT JOIN nrare_{split} n USING (entity_id)
            LEFT JOIN arare_{split} a USING (entity_id)
        )
        SELECT * FROM (
            SELECT entity_id, src, 'n1' AS kt, 'n1|' || country || '|' || n AS k
            FROM (SELECT *, unnest([n1, n2]) AS n FROM b) WHERE n IS NOT NULL
            UNION ALL
            SELECT entity_id, src, 'n2', 'n2|' || country || '|' || least(n1, n2) || ' ' || greatest(n1, n2)
            FROM b WHERE n2 IS NOT NULL
            UNION ALL
            SELECT entity_id, src, 'px', 'px|' || country || '|' || left(core_ns, 6) FROM b WHERE length(core_ns) >= 4
            UNION ALL
            SELECT entity_id, src, 'sk', 'sk|' || country || '|' || left(skel_ns, 8) FROM b WHERE length(skel_ns) >= 3
            UNION ALL
            SELECT entity_id, src, 'a2', 'a2|' || country || '|' || least(a1, a2) || ' ' || greatest(a1, a2)
            FROM b WHERE a2 IS NOT NULL
            UNION ALL
            SELECT entity_id, src, 'na', 'na|' || country || '|' || n || '|' || a
            FROM (SELECT entity_id, src, country, n, unnest([a1, a2]) AS a
                  FROM (SELECT *, unnest([n1, n2]) AS n FROM b))
            WHERE n IS NOT NULL AND a IS NOT NULL
        )
    """)
    con.execute(f"""
        CREATE OR REPLACE TABLE blocks_{split} AS
        SELECT k, any_value(kt) AS kt,
               count(*) FILTER (WHERE src = 1) AS n1, count(*) FILTER (WHERE src > 1) AS n23
        FROM keys_{split} GROUP BY k HAVING n1 > 0 AND n23 > 0
    """)
    n_keys = con.execute(f'SELECT count(*) FROM keys_{split}').fetchone()[0]
    print(f'  keys_{split}: {n_keys:,} rows, built with block stats in {time.time() - t:,.1f}s')
    return con.sql(f"""
        SELECT kt, count(*) AS keys,
               count(*) FILTER (WHERE n1 * n23 <= {MAX_KEY_PAIRS}) AS keys_kept,
               sum(n1 * n23) FILTER (WHERE n1 * n23 <= {MAX_KEY_PAIRS}) AS pairs_kept,
               sum(n1 * n23) FILTER (WHERE n1 * n23 > {MAX_KEY_PAIRS}) AS pairs_dropped,
               max(n1 * n23) AS largest_block
        FROM blocks_{split} GROUP BY kt ORDER BY kt
    """).df()


def build_candidates(split, out, left_ids=None):
    """out(id1, id2, n_keys, min_block, k_*): S1 × (S2 ∪ S3) pairs sharing at least one kept key."""
    left_filter = f'JOIN {left_ids} USING (entity_id)' if left_ids else ''
    flags = ', '.join(f"max((l.kt = '{kt}')::TINYINT) AS k_{kt}" for kt in KEY_TYPES)
    con.execute(f"""
        CREATE OR REPLACE TABLE {out} AS
        WITH b AS (SELECT k, n1 * n23 AS bp FROM blocks_{split} WHERE n1 * n23 <= {MAX_KEY_PAIRS}),
             l AS (SELECT entity_id, kt, k FROM keys_{split} {left_filter} WHERE src = 1),
             r AS (SELECT entity_id, k FROM keys_{split} WHERE src > 1)
        SELECT l.entity_id AS id1, r.entity_id AS id2,
               count(*) AS n_keys, min(b.bp) AS min_block,
               {flags}
        FROM l JOIN b USING (k) JOIN r USING (k)
        GROUP BY l.entity_id, r.entity_id
        QUALIFY row_number() OVER (PARTITION BY id1 ORDER BY count(*) DESC, min(b.bp), id2) <= {MAX_CANDS_PER_S1}
    """)
    n, n1 = con.execute(f'SELECT count(*), count(DISTINCT id1) FROM {out}').fetchone()
    print(f'{out}: {n:,} pairs for {n1:,} S1 records')


with stage('blocking keys (train)'):
    display(build_keys('train'))

### Train / validation split of S1 entities and blocking recall

Recall = share of ground-truth pairs of the validation entities that survive blocking. It is an upper bound on the
final recall. If it is too low, raise `MAX_KEY_PAIRS` (more pairs, slower); if candidates are too many, lower it.

In [ ]:
con.execute(f"""
    CREATE OR REPLACE TABLE gt AS
    SELECT DISTINCT trim(source1_entity_id) AS id1, trim(m) AS id2
    FROM (SELECT source1_entity_id, unnest(string_split(matched_entity_ids, ',')) AS m
          FROM read_csv('{GT_PATH}', delim='\\t', header=true, all_varchar=true))
    WHERE trim(m) <> ''
""")
n_s1 = con.execute('SELECT count(*) FROM recs_train WHERE src = 1').fetchone()[0]
n_val = min(N_VAL_S1, n_s1 // 5)
n_train = min(N_TRAIN_S1, n_s1 - n_val)
con.execute(f"""
    CREATE OR REPLACE TABLE s1_split AS
    SELECT entity_id, CASE WHEN rn <= {n_val} THEN 'val' ELSE 'train' END AS part
    FROM (SELECT entity_id, row_number() OVER (ORDER BY hash(entity_id || '{SEED}')) AS rn
          FROM recs_train WHERE src = 1)
    WHERE rn <= {n_val + n_train}
""")
con.execute("CREATE OR REPLACE TABLE ids_train AS SELECT entity_id FROM s1_split WHERE part = 'train'")
con.execute("CREATE OR REPLACE TABLE ids_val   AS SELECT entity_id FROM s1_split WHERE part = 'val'")
print(f'gt pairs: {con.execute("SELECT count(*) FROM gt").fetchone()[0]:,} | train S1: {n_train:,} | val S1: {n_val:,}')

with stage('candidates (train + val)'):
    build_candidates('train', 'cands_train', 'ids_train')
    build_candidates('train', 'cands_val', 'ids_val')

via = ', '.join(f'round(sum(c.k_{kt}) / count(*), 4) AS via_{kt}' for kt in KEY_TYPES)
recall = con.sql(f"""
    SELECT count(*) AS gt_pairs, count(c.id1) AS found, round(count(c.id1) / count(*), 4) AS blocking_recall, {via}
    FROM gt JOIN ids_val v ON gt.id1 = v.entity_id
    LEFT JOIN cands_val c ON c.id1 = gt.id1 AND c.id2 = gt.id2
""").df()
recall

## 4. Features and training

All string similarities run through `rapidfuzz.process.cpdist` (element-wise, in C, multi-threaded) instead of a
Python loop over pairs.

In [ ]:
FEATURE_SQL = """
    SELECT c.*, {label} AS label,
           a.name_norm AS n1, b.name_norm AS n2, a.name_core AS c1, b.name_core AS c2,
           a.name_skel AS sk1, b.name_skel AS sk2, a.addr_norm AS a1, b.addr_norm AS a2,
           a.addr_nums AS d1, b.addr_nums AS d2, b.src AS src2, (a.non_ascii OR b.non_ascii) AS non_ascii
    FROM {cands} c
    JOIN recs_{split} a ON a.entity_id = c.id1
    JOIN recs_{split} b ON b.entity_id = c.id2
    {gt_join}
    {where}
"""
FEATURES = ['name_ratio', 'name_tset', 'core_tsort', 'core_partial', 'core_jw', 'skel_ratio',
            'addr_tset', 'addr_tsort', 'num_tset', 'num_missing', 'core_empty', 'len_ratio',
            'n_keys', 'log_min_block', *[f'k_{kt}' for kt in KEY_TYPES], 'src2', 'non_ascii']


def featurize(df):
    def sim(scorer, x, y):
        return process.cpdist(df[x].tolist(), df[y].tolist(), scorer=scorer, workers=-1)

    out = pd.DataFrame({
        'name_ratio':   sim(fuzz.ratio, 'n1', 'n2'),
        'name_tset':    sim(fuzz.token_set_ratio, 'n1', 'n2'),
        'core_tsort':   sim(fuzz.token_sort_ratio, 'c1', 'c2'),
        'core_partial': sim(fuzz.partial_ratio, 'c1', 'c2'),
        'core_jw':      sim(JaroWinkler.normalized_similarity, 'c1', 'c2'),
        'skel_ratio':   sim(fuzz.ratio, 'sk1', 'sk2'),
        'addr_tset':    sim(fuzz.token_set_ratio, 'a1', 'a2'),
        'addr_tsort':   sim(fuzz.token_sort_ratio, 'a1', 'a2'),
        'num_tset':     sim(fuzz.token_set_ratio, 'd1', 'd2'),
    }, index=df.index)
    l1, l2 = df['c1'].str.len(), df['c2'].str.len()
    out['num_missing'] = (df['d1'] == '').astype(np.int8) + (df['d2'] == '').astype(np.int8)
    out['core_empty'] = ((l1 == 0) | (l2 == 0)).astype(np.int8)
    out['len_ratio'] = (np.minimum(l1, l2) / np.maximum(np.maximum(l1, l2), 1)).astype(np.float32)
    out['log_min_block'] = np.log1p(df['min_block']).astype(np.float32)
    for col in ['n_keys', *[f'k_{kt}' for kt in KEY_TYPES], 'src2']:
        out[col] = df[col].astype(np.int16)
    out['non_ascii'] = df['non_ascii'].astype(np.int8)
    return out[FEATURES]


def stream_features(cands, split, with_label, neg_rate=1.0):
    """Yield (Arrow batch with id1/id2/label, feature frame) in batches of BATCH_ROWS pairs.
    neg_rate < 1 keeps that (deterministic) fraction of negatives, before any feature is computed.

    DuckDB's memory limit is lowered while streaming: its join tables otherwise held ~7 GB for the whole
    test stage, and together with the Python batches that crashed a 12.7 GB Colab runtime."""
    where = (f"WHERE g.id1 IS NOT NULL OR hash(c.id1 || c.id2 || '{SEED}') % 1000000 < {int(neg_rate * 1e6)}"
             if neg_rate < 1 else '')
    sql = FEATURE_SQL.format(
        cands=cands, split=split, where=where,
        label='(g.id1 IS NOT NULL)' if with_label else 'NULL::BOOLEAN',
        gt_join='LEFT JOIN gt g ON g.id1 = c.id1 AND g.id2 = c.id2' if with_label else '')
    con.execute(f"SET memory_limit = '{DUCKDB_STREAM_MEMORY}'")
    try:
        res = con.execute(sql)
        reader = (res.to_arrow_reader if hasattr(res, 'to_arrow_reader') else res.fetch_record_batch)(BATCH_ROWS)
        for batch in reader:
            yield batch, featurize(batch.to_pandas())
    finally:
        con.execute(f"SET memory_limit = '{DUCKDB_MEMORY}'")


def load_features(cands, split, neg_rate=1.0):
    """(X, y) for a labelled candidate table; only labels are kept besides features, to save memory."""
    X, y = [], []
    for batch, feats in stream_features(cands, split, with_label=True, neg_rate=neg_rate):
        X.append(feats)
        y.append(batch.column('label').to_numpy(zero_copy_only=False).astype(bool))
    return pd.concat(X, ignore_index=True), np.concatenate(y)


n_all, n_pos = con.execute("""
    SELECT count(*), count(g.id1) FROM cands_train c LEFT JOIN gt g ON g.id1 = c.id1 AND g.id2 = c.id2
""").fetchone()
neg_rate = min(1.0, max(MAX_TRAIN_ROWS - n_pos, 0) / max(n_all - n_pos, 1))
print(f'train candidates: {n_all:,}  positives: {n_pos:,}  negatives kept: {neg_rate:.1%}')

with stage('features (train)'):
    X_tr, y_tr = load_features('cands_train', 'train', neg_rate)
print(f'training rows: {len(y_tr):,}')

with stage('train model'):
    model = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, max_leaf_nodes=63,
                                           early_stopping=True, validation_fraction=0.1, random_state=SEED)
    model.fit(X_tr, y_tr)
print('boosting iterations:', model.n_iter_)
del X_tr, y_tr

## 5. Validation and threshold

Pair-level F-beta on the held-out S1 entities. Ground-truth pairs that blocking never produced count as false
negatives, so this is an honest end-to-end estimate.

In [ ]:
with stage('features + predict (val)'):
    X_val, y_val = load_features('cands_val', 'train')
    p_val = model.predict_proba(X_val)[:, 1]
total_pos = con.execute('SELECT count(*) FROM gt JOIN ids_val v ON gt.id1 = v.entity_id').fetchone()[0]

order = np.argsort(-p_val)
tp = np.cumsum(y_val[order])
fp = np.cumsum(~y_val[order])
fn = total_pos - tp
b2 = BETA ** 2
fbeta = (1 + b2) * tp / np.maximum((1 + b2) * tp + b2 * fn + fp, 1)
best = int(np.argmax(fbeta))
THRESHOLD = float(p_val[order][best])
print(f'best F{BETA:g} = {fbeta[best]:.4f} at threshold {THRESHOLD:.4f}')
print(f'precision = {tp[best] / (tp[best] + fp[best]):.4f}  recall = {tp[best] / total_pos:.4f} '
      f'(blocking recall ceiling = {y_val.sum() / total_pos:.4f})')

joblib.dump({'model': model, 'threshold': THRESHOLD, 'features': FEATURES,
             'max_key_pairs': MAX_KEY_PAIRS, 'max_cands_per_s1': MAX_CANDS_PER_S1},
            WORK_DIR / 'er_model.joblib')
save_to_cache(WORK_DIR / 'er_model.joblib')
del X_val, y_val, p_val

## 6. Predict on test

Features are computed and scored batch by batch, so memory stays flat regardless of the number of candidates.
Output uses the ground-truth layout (`source1_entity_id`, comma-separated `matched_entity_ids`); check it against
the README / `utils/validate_submission.py` before submitting.

In [ ]:
with stage('blocking + candidates (test)'):
    display(build_keys('test'))
    build_candidates('test', 'cands_test')

kept = []  # predicted pairs stay as compact Arrow tables, not millions of Python strings
with stage('features + predict (test)'):
    for batch, X in stream_features('cands_test', 'test', with_label=False):
        keep = pa.array(model.predict_proba(X)[:, 1] >= THRESHOLD)
        kept.append(pa.table({'id1': batch.column('id1'), 'id2': batch.column('id2')}).filter(keep))
matches = pa.concat_tables(kept)
del kept
print(f'predicted pairs: {matches.num_rows:,}')

s1_order = pd.read_csv(SOURCES['test'][1], sep='\t', dtype=str, usecols=['entity_id'])['entity_id'].str.strip()
s1_order = pa.table({'entity_id': s1_order.to_numpy(), 'pos': np.arange(len(s1_order))})
submission = con.execute("""
    SELECT s.entity_id AS source1_entity_id,
           coalesce(string_agg(m.id2, ',' ORDER BY m.id2), '') AS matched_entity_ids
    FROM s1_order s LEFT JOIN matches m ON m.id1 = s.entity_id
    GROUP BY s.entity_id, s.pos ORDER BY s.pos
""").df()
out_path = WORK_DIR / 'test_predictions.tsv'
submission.to_csv(out_path, sep='\t', index=False)
out_path = save_to_cache(out_path)
print(out_path, submission.shape)
n_matches = submission['matched_entity_ids'].map(lambda s: len(s.split(',')) if s else 0)
print(n_matches.value_counts().sort_index().rename('S1 records by number of predicted matches'))
submission.head()